# 01 — Road Graph EDA
Explore the synthetic city road network structure, topology metrics, and bottleneck analysis.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from src.simulation_engine.road_graph import CityRoadGraph
from src.graph_analysis.graph_metrics import GraphAnalyzer

# Build graph
rg = CityRoadGraph(seed=42)
rg.build_synthetic(num_nodes=40, grid_size=50)
G = rg.G
print(f'Nodes: {G.number_of_nodes()}  Edges: {G.number_of_edges()}')

In [ ]:
# Visualise road network
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

pos = {n: (d['x'], d['y']) for n, d in G.nodes(data=True)}
zone_colors = {'residential':'#4CAF50','commercial':'#2196F3','industrial':'#FF9800','campus':'#9C27B0','transit':'#00BCD4'}
node_colors = [zone_colors.get(G.nodes[n].get('zone_type','residential'), '#888') for n in G.nodes()]
node_sizes  = [200 if G.nodes[n].get('is_major_intersection') else 60 for n in G.nodes()]

nx.draw(G, pos, ax=axes[0], node_color=node_colors, node_size=node_sizes,
        edge_color='#cccccc', width=0.5, with_labels=False, arrows=False)
axes[0].set_title('City Road Network (color = zone type)', fontsize=12)

# Legend
from matplotlib.patches import Patch
legend = [Patch(color=c, label=z) for z, c in zone_colors.items()]
axes[0].legend(handles=legend, loc='lower left', fontsize=9)

# Degree distribution
degrees = [d for _, d in G.degree()]
axes[1].hist(degrees, bins=range(1, max(degrees)+2), color='#2196F3', edgecolor='white', rwidth=0.8)
axes[1].set_xlabel('Node Degree', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('Degree Distribution', fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/road_graph_eda.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Bottleneck analysis
analyzer = GraphAnalyzer(rg)
bc = analyzer.compute_betweenness_centrality()
report = analyzer.full_report()

print('=== Graph Statistics ===')
for k, v in report['graph_stats'].items():
    print(f'  {k:35s}: {v}')

print('\n=== Top Bottleneck Nodes ===')
for item in report['top_bottlenecks']:
    n = item['node']
    zone = G.nodes[n].get('zone_type', '?')
    print(f'  Node {n:3d} | zone={zone:12s} | centrality={item["score"]:.4f}')

print('\n=== Infrastructure Recommendations ===')
for rec in report['recommendations'][:3]:
    print(f'  [{rec["priority"]}] {rec["type"]}: {rec["suggestion"]}')

In [ ]:
# Betweenness centrality visualisation
fig, ax = plt.subplots(figsize=(10, 9))
bc_vals = np.array([bc.get(n, 0) for n in G.nodes()])
sizes = 50 + 600 * bc_vals / (bc_vals.max() + 1e-8)
nx.draw(G, pos, ax=ax, node_color=bc_vals, node_size=sizes,
        edge_color='#dddddd', width=0.6, cmap=plt.cm.YlOrRd,
        with_labels=False, arrows=False)
sm = plt.cm.ScalarMappable(cmap=plt.cm.YlOrRd,
                             norm=plt.Normalize(bc_vals.min(), bc_vals.max()))
plt.colorbar(sm, ax=ax, label='Betweenness Centrality')
ax.set_title('Road Network Bottleneck Analysis\n(larger/darker = higher centrality = critical node)', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/bottleneck_analysis.png', dpi=120, bbox_inches='tight')
plt.show()